In [1]:
#Part 1
#Step 1
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
import math
import time
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
plt.switch_backend('agg')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SOS_token = 0
EOS_token = 1
MAX_LENGTH = 10

In [2]:
#Step 2
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2 # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [3]:
#Step 3
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )


def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()


def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    lines = open(
        'data/%s-%s.txt' % (lang1, lang2),
        encoding='utf-8'
    ).read().strip().split('\n')

    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs


eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


def filterPair(p):
    return (
        len(p[0].split(' ')) < MAX_LENGTH and
        len(p[1].split(' ')) < MAX_LENGTH and
        p[1].startswith(eng_prefixes)
    )


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]


In [4]:
#Step 4
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])

    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    
    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        
    train_data = TensorDataset(torch.LongTensor(input_ids).to(device), torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler,batch_size=batch_size)
    return input_lang, output_lang, train_dataloader
# Build the dataset
batch_size = 32
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counted words:
fra 4601
eng 2991


In [5]:
#Part 2
#Step 1
class EncoderRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden # output: [B, T, H], hidden: [1, B, H]

In [6]:
#Step 2
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)


    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):

        batch_size = encoder_outputs.size(0)

        decoder_input = torch.empty(
            batch_size, 1,
            dtype=torch.long,
            device=device
        ).fill_(SOS_token)

        decoder_hidden = encoder_hidden

        decoder_outputs = []

        for i in range(MAX_LENGTH):

            decoder_output, decoder_hidden = self.forward_step(
                decoder_input,
                decoder_hidden
            )

            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)

            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)

        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        return decoder_outputs, decoder_hidden, None
    
    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden

In [7]:
#Step 3
hidden_size = 128

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_no_attn = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Encoder:")
print(encoder)
print(f"\nEncoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")

print("\nDecoder (no attention):")
print(decoder_no_attn)
print(f"\nDecoder parameters: {sum(p.numel() for p in decoder_no_attn.parameters()):,}")

Encoder:
EncoderRNN(
  (embedding): Embedding(4601, 128)
  (gru): GRU(128, 128, batch_first=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

Encoder parameters: 688,000

Decoder (no attention):
DecoderRNN(
  (embedding): Embedding(2991, 128)
  (gru): GRU(128, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=2991, bias=True)
)

Decoder parameters: 867,759


In [8]:
#Part 3
#Step 1
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size) # projects decoder state (query)
        self.Ua = nn.Linear(hidden_size, hidden_size) # projects encoder states (keys)
        self.Va = nn.Linear(hidden_size, 1) # reduces to scalar score
        
    def forward(self, query, keys):
        # query: decoder hidden state [batch, 1, hidden_size]
        # keys: encoder outputs [batch, seq_len, hidden_size]
        #
        # Returns:
        # context: weighted sum [batch, 1, hidden_size]
        # weights: attention weights [batch, 1, seq_len]
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        # scores: [batch, seq_len, 1]
        
        scores = scores.squeeze(2).unsqueeze(1)
        # scores: [batch, 1, seq_len]

        weights = F.softmax(scores, dim=-1)
        # weights: [batch, 1, seq_len]

        context = torch.bmm(weights, keys)
        # context: [batch, 1, hidden_size]

        return context, weights

In [9]:
#Step 2
# Create a dummy attention module
attn = BahdanauAttention(hidden_size).to(device)

# Simulate encoder outputs and a decoder hidden state
dummy_encoder_outputs = torch.randn(batch_size, MAX_LENGTH,
hidden_size).to(device)
dummy_decoder_hidden = torch.randn(batch_size, 1, hidden_size).to(device)

context, weights = attn(dummy_decoder_hidden, dummy_encoder_outputs)

print(f"Query shape (decoder hidden): {dummy_decoder_hidden.shape}")
print(f"Keys shape (encoder outputs): {dummy_encoder_outputs.shape}")
print(f"Context vector shape: {context.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Attention weights sum: {weights.sum(dim=-1)}")

Query shape (decoder hidden): torch.Size([32, 1, 128])
Keys shape (encoder outputs): torch.Size([32, 10, 128])
Context vector shape: torch.Size([32, 1, 128])
Attention weights shape: torch.Size([32, 1, 10])
Attention weights sum: tensor([[1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000]], grad_fn=<SumBackward1>)


In [10]:
#Step 2
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)
    
    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long,
        device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []
        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
            decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)
            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)
        return decoder_outputs, decoder_hidden, attentions
    
    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))
        query = hidden.permute(1, 0, 2) # [1, B, H] -> [B, 1, H]
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2) # [B, 1, 2*H]
        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)
        return output, hidden, attn_weights
    
decoder_attn = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)
print(f"Decoder WITHOUT attention: {sum(p.numel() for p in decoder_no_attn.parameters()):,} params")
print(f"Decoder WITH attention: {sum(p.numel() for p in decoder_attn.parameters()):,} params")
print(f"\nExtra parameters from attention: " f"{sum(p.numel() for p in decoder_attn.parameters()) - sum(p.numel() for p in decoder_no_attn.parameters()):,}")

Decoder WITHOUT attention: 867,759 params
Decoder WITH attention: 950,064 params

Extra parameters from attention: 82,305


In [11]:
#Part 4
#Step 1
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

def train_epoch(dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden,target_tensor)

        loss = criterion(
        decoder_outputs.view(-1, decoder_outputs.size(-1)),
        target_tensor.view(-1)
        )

        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(dataloader)

def showPlot(points, title="Training Loss"):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)
    plt.title(title)
    plt.xlabel("Epoch (x5)")
    plt.ylabel("Loss")
    plt.savefig("training_loss.png")
    plt.show()

In [12]:
#Step 2
def train_model(train_dataloader, encoder, decoder, n_epochs,learning_rate=0.001, print_every=5, plot_every=5):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
        encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss
        
        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs), epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0
    showPlot(plot_losses)
    return plot_losses

In [13]:
#Step 3
# Re-initialize encoder and attention decoder
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_attn = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Training Seq2Seq WITH Bahdanau Attention...")
print("=" * 50)
attn_losses = train_model(train_dataloader, encoder, decoder_attn,n_epochs=80, print_every=5, plot_every=5)

Training Seq2Seq WITH Bahdanau Attention...
1m 27s (- 21m 56s) (5 6%) 1.5406
2m 59s (- 20m 54s) (10 12%) 0.6822
4m 19s (- 18m 42s) (15 18%) 0.3542
5m 38s (- 16m 54s) (20 25%) 0.1978
6m 55s (- 15m 14s) (25 31%) 0.1231
8m 9s (- 13m 36s) (30 37%) 0.0855
9m 22s (- 12m 2s) (35 43%) 0.0648
10m 34s (- 10m 34s) (40 50%) 0.0528
11m 48s (- 9m 11s) (45 56%) 0.0458
13m 1s (- 7m 48s) (50 62%) 0.0414
14m 13s (- 6m 27s) (55 68%) 0.0374
15m 26s (- 5m 8s) (60 75%) 0.0351
16m 39s (- 3m 50s) (65 81%) 0.0336
17m 54s (- 2m 33s) (70 87%) 0.0310
19m 11s (- 1m 16s) (75 93%) 0.0305
20m 28s (- 0m 0s) (80 100%) 0.0296


C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\2489845712.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
#Part 5
#Step 1
def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1,-1)

def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # p.cyan("Encoder outputs:", encoder_outputs)
        # p.yellow("encoder_hidden:", encoder_hidden)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(
        encoder_outputs, encoder_hidden
        )
        # p.yellow("encoder_hidden:", encoder_hidden)
        
        # p.red("Decoder hidden:", decoder_hidden)
        
        # p.green("Decoder outputs:", decoder_outputs)
        # p.blue("Decoder attentions:", decoder_attn)
        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()
        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

def evaluateRandomly(encoder, decoder, pairs, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0],input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

encoder.eval()
decoder_attn.eval()
# Re-load pairs for evaluation
_, _, pairs = prepareData('eng', 'fra', True)
pairs = filterPairs(pairs)
evaluateRandomly(encoder, decoder_attn, pairs)


Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counted words:
fra 4601
eng 2991
> je ne suis pas assez bon pour toi
= i m not good enough for you
< i m not good enough for you <EOS>

> tu es liberee de toute responsabilite
= you re free of all responsibility
< you re free of all responsibility <EOS>

> je viens de portugal
= i am from portugal
< i am from portugal at a cold fish <EOS>

> je ne vais pas disparaitre
= i m not going to disappear
< i m not going to disappear <EOS>

> ils cherchent un bouc emissaire
= they re looking for a scapegoat
< they re looking for a scapegoat <EOS>

> je suis musicienne
= i m a musician
< i am a stranger here <EOS>

> elle est bon ecrivain
= she is a good writer
< she is a good writer <EOS>

> vous etes trop stupide pour vivre
= you re too stupid to live
< you re too stupid to live <EOS>

> je ne sais pas quoi faire
= i m in way over my head
< i m in way over my head <EOS>

> il est populaire parmi nous
= he is popular am

In [15]:
#Step 2
def showAttention(input_sentence, output_words, attentions):
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111)

    attn = attentions.cpu().numpy()

    cax = ax.matshow(attn, cmap='viridis')
    fig.colorbar(cax)

    input_tokens = input_sentence.split(' ') + ['<EOS>']
    output_tokens = output_words

    ax.set_xticks(range(len(input_tokens)))
    ax.set_yticks(range(len(output_tokens)))

    ax.set_xticklabels(input_tokens, rotation=90)
    ax.set_yticklabels(output_tokens)

    plt.xlabel("Source (French)")
    plt.ylabel("Target (English)")
    plt.title("Bahdanau Attention Alignment")

    plt.tight_layout()

    plt.savefig("attention_heatmap.png")
    plt.show()
def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate(
        encoder,
        decoder_attn,
        input_sentence,
        input_lang,
        output_lang
    )

    print('input =', input_sentence)
    print('output =', ' '.join(output_words))

    showAttention(
        input_sentence,
        output_words,
        attentions[0, :len(output_words), :]
    )
    
evaluateAndShowAttention('il n est pas aussi grand que son pere')
evaluateAndShowAttention('je suis trop fatigue pour conduire')
evaluateAndShowAttention('je suis desole si c est une question idiote')
evaluateAndShowAttention('je suis reellement fiere de vous')

input = il n est pas aussi grand que son pere
output = he is not as tall as his father <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


input = je suis trop fatigue pour conduire
output = i am too tired to drive <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


input = je suis desole si c est une question idiote
output = i m sorry if this is a stupid question <EOS>
input = je suis reellement fiere de vous
output = i m really proud of you <EOS>


C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\1702714426.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
                                            #HW 6
#Task 1

#Створюємо encoder та decoder без механізму уваги
encoder_v = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder_v = DecoderRNN(hidden_size, output_lang.n_words).to(device)

print("Training Seq2Seq WITHOUT Attention")

#Навчання vanilla Seq2Seq моделі
#Модель тренується 80 епох для порівняння з attention моделлю
vanilla_losses = train_model(
    train_dataloader,
    encoder_v,
    decoder_v,
    n_epochs=80,
    print_every=5,
    plot_every=5
)

#Evaluation
print("Evaluation WITH Attention")
evaluateRandomly(encoder, decoder_attn, pairs)

print("Evaluation WITHOUT Attention")
evaluateRandomly(encoder_v, decoder_v, pairs)

#Loss comparison
plt.figure()
plt.plot(attn_losses, label="Attention")
plt.plot(vanilla_losses, label="Vanilla")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Comparison")
plt.show()

Training Seq2Seq WITHOUT Attention
0m 53s (- 13m 26s) (5 6%) 1.6847
1m 46s (- 12m 28s) (10 12%) 0.9023
2m 40s (- 11m 36s) (15 18%) 0.5691
3m 34s (- 10m 42s) (20 25%) 0.3730
4m 27s (- 9m 49s) (25 31%) 0.2524
5m 21s (- 8m 55s) (30 37%) 0.1758
6m 15s (- 8m 2s) (35 43%) 0.1271
7m 8s (- 7m 8s) (40 50%) 0.0965
8m 2s (- 6m 15s) (45 56%) 0.0761
8m 55s (- 5m 21s) (50 62%) 0.0621
9m 51s (- 4m 28s) (55 68%) 0.0531
10m 44s (- 3m 34s) (60 75%) 0.0474
11m 37s (- 2m 41s) (65 81%) 0.0433
12m 31s (- 1m 47s) (70 87%) 0.0387
13m 25s (- 0m 53s) (75 93%) 0.0366
14m 19s (- 0m 0s) (80 100%) 0.0341
Evaluation WITH Attention
> il est en mauvaise sante
= he is in poor health
< he is in good health <EOS>

> elle est privee de sortie
= she is forbidden to go out
< she is forbidden to go out <EOS>

> je dors peu
= i m a light sleeper
< i m a little over <EOS>

> je suis en bonne sante
= i m healthy
< i m healthy <EOS>

> vous etes tres braves
= you re very brave
< you re very brave <EOS>

> nous sommes a court d a

C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\2489845712.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\1733164219.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
#Task 2
class LuongAttention(nn.Module):

    def __init__(self, hidden_size, method='dot'):
        super(LuongAttention, self).__init__()

        #Тип attention (dot або general)
        self.method = method

        #Для методу general використовується лінійне перетворення
        if method == "general":
            self.W = nn.Linear(hidden_size, hidden_size)

    def score(self, query, keys):

        #query = hidden state декодера
        #keys = encoder outputs

        if self.method == "dot":
            #dot product attention
            #s_i^T * h_j
            return torch.bmm(query, keys.transpose(1,2))

        elif self.method == "general":
            #general attention
            #s_i^T * W * h_j
            query = self.W(query)
            return torch.bmm(query, keys.transpose(1,2))

    def forward(self, query, keys):

        #Обчислення attention scores
        scores = self.score(query, keys)

        #Перетворення scores у attention weights
        weights = F.softmax(scores, dim=-1)

        #Обчислення context vector
        #context = weighted sum encoder outputs
        context = torch.bmm(weights, keys)

        return context, weights


#Тестування реалізації Luong Attention

luong_attn = LuongAttention(hidden_size).to(device)

dummy_encoder_outputs = torch.randn(batch_size, MAX_LENGTH, hidden_size).to(device)
dummy_decoder_hidden = torch.randn(batch_size, 1, hidden_size).to(device)

context, weights = luong_attn(dummy_decoder_hidden, dummy_encoder_outputs)

print("Context shape:", context.shape)
print("Weights shape:", weights.shape)

Context shape: torch.Size([32, 1, 128])
Weights shape: torch.Size([32, 1, 10])


In [18]:
#Task 3
class BiEncoderRNN(nn.Module):

    def __init__(self, input_size, hidden_size):

        super(BiEncoderRNN, self).__init__()

        self.hidden_size = hidden_size

        #embedding перетворює слова у векторне представлення
        self.embedding = nn.Embedding(input_size, hidden_size)

        #GRU працює у двонаправленому режимі
        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, x):

        #Перетворення токенів у embeddings
        embedded = self.embedding(x)

        #Передача embeddings у GRU
        outputs, hidden = self.gru(embedded)

        #Після bidirectional GRU отримуємо
        #два hidden states (forward + backward)

        #Їх потрібно об'єднати
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, :, self.hidden_size:]

        return outputs, hidden


#Тестування Bidirectional Encoder

bi_encoder = BiEncoderRNN(input_lang.n_words, hidden_size).to(device)

dummy_input = torch.randint(0, input_lang.n_words, (batch_size, MAX_LENGTH)).to(device)

outputs, hidden = bi_encoder(dummy_input)

print("Encoder outputs shape:", outputs.shape)
print("Hidden shape:", hidden.shape)

Encoder outputs shape: torch.Size([32, 10, 128])
Hidden shape: torch.Size([2, 32, 128])


In [19]:
#Task 4
import scipy.stats
import numpy as np
import matplotlib.pyplot as plt

entropy_positions = []
entropy_values = []

num_samples = 50

for i in range(num_samples):

    pair = random.choice(pairs)

    output_words, attentions = evaluate(
        encoder,
        decoder_attn,
        pair[0],
        input_lang,
        output_lang
    )

    attentions = attentions.detach().cpu().numpy()

    #attentions shape = [decoder_len, encoder_len]

    for step in range(attentions.shape[0]):

        weights = attentions[step]

        #Робимо weights 1D
        weights = np.ravel(weights)

        #Нормалізація (entropy очікує distribution)
        weights = weights / np.sum(weights)

        #Обчислення entropy
        entropy = scipy.stats.entropy(weights)

        #Гарантуємо float
        entropy = float(np.asarray(entropy).mean())

        entropy_positions.append(step)
        entropy_values.append(entropy)


#Побудова графіку
plt.figure(figsize=(8,5))

plt.scatter(entropy_positions, entropy_values, alpha=0.5)

plt.xlabel("Decoder step")
plt.ylabel("Attention entropy")

plt.title("Attention entropy vs decoder step")

plt.show()

C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\3078243072.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
#Task 5
#Значення для експерименту
teacher_forcing_ratios = [0.0, 0.5, 1.0]

#Словник для збереження результатів
results = {}

for ratio in teacher_forcing_ratios:

    print("======================================")
    print("Training with teacher_forcing_ratio =", ratio)

    #Встановлюємо значення teacher forcing
    teacher_forcing_ratio = ratio

    #Запускаємо навчання моделі
    losses = train_model(
        train_dataloader,
        encoder,
        decoder_attn,
        n_epochs=20
    )

    #Зберігаємо фінальне значення loss
    final_loss = losses[-1]

    results[ratio] = final_loss

    print("Final loss:", final_loss)


#Виводимо результати експерименту
print("\n===== Порівняння результатів =====")

for ratio, loss in results.items():

    print("teacher_forcing_ratio =", ratio, " -> final loss =", loss)

Training with teacher_forcing_ratio = 0.0
1m 13s (- 3m 40s) (5 25%) 0.0227
2m 27s (- 2m 27s) (10 50%) 0.0197
3m 41s (- 1m 13s) (15 75%) 0.0195
4m 51s (- 0m 0s) (20 100%) 0.0179
Final loss: 0.017880476576012074
Training with teacher_forcing_ratio = 0.5


C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\2489845712.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


1m 9s (- 3m 27s) (5 25%) 0.0185
2m 20s (- 2m 20s) (10 50%) 0.0173
3m 30s (- 1m 10s) (15 75%) 0.0183
4m 41s (- 0m 0s) (20 100%) 0.0160
Final loss: 0.015959151069761784
Training with teacher_forcing_ratio = 1.0
1m 10s (- 3m 31s) (5 25%) 0.0179
2m 22s (- 2m 22s) (10 50%) 0.0161
3m 33s (- 1m 11s) (15 75%) 0.0188
4m 45s (- 0m 0s) (20 100%) 0.0154
Final loss: 0.015448291438183612

===== Порівняння результатів =====
teacher_forcing_ratio = 0.0  -> final loss = 0.017880476576012074
teacher_forcing_ratio = 0.5  -> final loss = 0.015959151069761784
teacher_forcing_ratio = 1.0  -> final loss = 0.015448291438183612


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import numpy as np
import math
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SOS_token = 0
EOS_token = 1
PAD_token = 2
MAX_LENGTH = 10
        
batch_size = 32


Using device: cpu


In [22]:
#Part 2 | Step 1
class PositionalEncoding(nn.Module):
 """
 Adds positional information to token embeddings using sine and cosine
functions.
 """
 def __init__(self, d_model, max_len=5000, dropout=0.1):
    super(PositionalEncoding, self).__init__()
    self.dropout = nn.Dropout(p=dropout)

    # Create a matrix of shape (max_len, d_model) for positional encodings
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len,dtype=torch.float).unsqueeze(1)
    # Compute the div_term for sine and cosine
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

    # Apply sine to even indices (0, 2, 4, ...)
    pe[:, 0::2] = torch.sin(position * div_term)
    # Apply cosine to odd indices (1, 3, 5, ...)
    pe[:, 1::2] = torch.cos(position * div_term)

    # Add batch dimension: (max_len, d_model) -> (1, max_len, d_model)
    pe = pe.unsqueeze(0)

    # Register as buffer (not a parameter, but part of the model state)
    self.register_buffer('pe', pe)

 def forward(self, x):
    # x: [batch_size, seq_len, d_model]
    # Add positional encoding to input embeddings
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

d_model = 128
pe_module = PositionalEncoding(d_model, max_len=MAX_LENGTH)

# Create dummy embeddings
dummy_embeddings = torch.randn(batch_size, MAX_LENGTH, d_model)
encoded = pe_module(dummy_embeddings)
print(f"Input shape: {dummy_embeddings.shape}")
print(f"After PE: {encoded.shape}")

# Visualize positional encoding pattern
plt.figure(figsize=(10, 4))
plt.imshow(pe_module.pe[0, :50, :].numpy(), cmap='viridis', aspect='auto')
plt.colorbar()
plt.xlabel("Embedding Dimension")
plt.ylabel("Position")
plt.title("Positional Encoding Pattern (first 50 positions)")
plt.savefig("positional_encoding.png")
plt.show()

Input shape: torch.Size([32, 10, 128])
After PE: torch.Size([32, 10, 128])


C:\Users\prosi\AppData\Local\Temp\ipykernel_27264\1771691298.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
#Step 2
def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
 """
 Compute scaled dot-product attention.

 Args:
 query: [batch_size, num_heads, seq_len_q, d_k]
 key: [batch_size, num_heads, seq_len_k, d_k]
 value: [batch_size, num_heads, seq_len_v, d_v]
 mask: [batch_size, 1, 1, seq_len_k] or broadcastable shape
 dropout: nn.Dropout module (optional)

 Returns:
 output: [batch_size, num_heads, seq_len_q, d_v]
 attn_weights: [batch_size, num_heads, seq_len_q, seq_len_k]
 """
 d_k = query.size(-1)

 # Compute attention scores: Q @ K^T / sqrt(d_k)
 scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

 # Apply mask (if provided) - mask out padding tokens
 if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)

 # Apply softmax to get attention weights
 attn_weights = F.softmax(scores, dim=-1)

 # Apply dropout
 if dropout is not None:
    attn_weights = dropout(attn_weights)

 # Multiply weights by values
 output = torch.matmul(attn_weights, value)

 return output, attn_weights

# Test with dummy data
batch_size = 2 # кількість прикладів, 
num_heads = 4 # кількість голів уваги
seq_len = 5 # довжина послідовності
d_k = 16 # розмірність вектора для кожної голови

query = torch.randn(batch_size, num_heads, seq_len, d_k)
key = torch.randn(batch_size, num_heads, seq_len, d_k)
value = torch.randn(batch_size, num_heads, seq_len, d_k)

output, weights = scaled_dot_product_attention(query, key, value)

print(f"Query shape: {query.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Weights sum (should be 1.0): {weights.sum(dim=-1)[0, 0, 0].item():.4f}")


Query shape: torch.Size([2, 4, 5, 16])
Output shape: torch.Size([2, 4, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])
Weights sum (should be 1.0): 1.0000


In [24]:
#Step 3
class MultiHeadAttention(nn.Module):
 """
 Multi-Head Attention module.

 Projects Q, K, V into num_heads subspaces, computes attention in
parallel,
 then concatenates and projects the results.
 """
 def __init__(self, d_model, num_heads, dropout=0.1):
    super(MultiHeadAttention, self).__init__()
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads
    # Linear projections for Q, K, V
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)

    # Output projection
    self.W_o = nn.Linear(d_model, d_model)

    self.dropout = nn.Dropout(dropout)

 def split_heads(self, x):
    """
    Split the last dimension into (num_heads, d_k).
    Input: [batch_size, seq_len, d_model]
    Output: [batch_size, num_heads, seq_len, d_k]
    """
    batch_size, seq_len, d_model = x.size()
    return x.view(batch_size, seq_len, self.num_heads,self.d_k).transpose(1, 2)

 def combine_heads(self, x):
    """
    Combine heads back into original shape.
    Input: [batch_size, num_heads, seq_len, d_k]
    Output: [batch_size, seq_len, d_model]
    """
    batch_size, num_heads, seq_len, d_k = x.size()
    return x.transpose(1, 2).contiguous().view(batch_size, seq_len,self.d_model)
    
 def forward(self, query, key, value, mask=None):
    """
    Args:
    query: [batch_size, seq_len_q, d_model]
    key: [batch_size, seq_len_k, d_model]
    value: [batch_size, seq_len_v, d_model]
    mask: [batch_size, 1, seq_len_k] or [batch_size, seq_len_q,seq_len_k]
    """
    batch_size = query.size(0)

    # 1. Linear projections
    Q = self.W_q(query) # [batch_size, seq_len_q, d_model]
    K = self.W_k(key)
    V = self.W_v(value)

    # 2. Split into multiple heads
    Q = self.split_heads(Q) # [batch_size, num_heads, seq_len_q, d_k]
    K = self.split_heads(K)
    V = self.split_heads(V)

    # 3. Apply attention
    if mask is not None:
        mask = mask.unsqueeze(1) # Add headimension
    
    attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask, dropout=self.dropout)

    # 4. Combine heads
    attn_output = self.combine_heads(attn_output)

    # 5. Final linear projection
    output = self.W_o(attn_output)

    return output, attn_weights   

d_model = 128
num_heads = 8
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads).to(device)

# Dummy inputs
query = torch.randn(batch_size, seq_len, d_model).to(device)
key = torch.randn(batch_size, seq_len, d_model).to(device)
value = torch.randn(batch_size, seq_len, d_model).to(device)

output, weights = mha(query, key, value)

print(f"MHA output shape: {output.shape}")
print(f"MHA attention weights shape: {weights.shape}")
print(f"Number of parameters: {sum(p.numel() for p in mha.parameters()):,}")

    

MHA output shape: torch.Size([2, 10, 128])
MHA attention weights shape: torch.Size([2, 8, 10, 10])
Number of parameters: 66,048


In [25]:
#Step 4
class PositionWiseFeedForward(nn.Module):
 """
 Two-layer feed-forward network with ReLU activation.
 """
 def __init__(self, d_model, d_ff, dropout=0.1):
    super(PositionWiseFeedForward, self).__init__()
    self.fc1 = nn.Linear(d_model, d_ff)
    self.fc2 = nn.Linear(d_ff, d_model)
    self.dropout = nn.Dropout(dropout)

 def forward(self, x):
    # x: [batch_size, seq_len, d_model]
    return self.fc2(self.dropout(F.relu(self.fc1(x))))

In [26]:
#Step 5
class EncoderLayer(nn.Module):
 """
 Single Transformer encoder layer.
 """
 def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super(EncoderLayer, self).__init__()

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)

    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

 def forward(self, x, mask=None):
    # x: [batch_size, seq_len, d_model]

    # Self-attention sub-layer
    attn_output, _ = self.self_attn(x, x, x, mask)
    x = self.norm1(x + self.dropout1(attn_output))
    # Feed-forward sub-layer
    ff_output = self.feed_forward(x)
    x = self.norm2(x + self.dropout2(ff_output))
    
    return x


In [27]:
#Step 6
class DecoderLayer(nn.Module):
 """
 Single Transformer decoder layer.
 """
 def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super(DecoderLayer, self).__init__()

    self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
    self.feed_forward = PositionWiseFeedForward(d_model, d_ff,dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.norm3 = nn.LayerNorm(d_model)

    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)
    self.dropout3 = nn.Dropout(dropout)

 def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
    # x: [batch_size, tgt_seq_len, d_model]
    # encoder_output: [batch_size, src_seq_len, d_model]
    
    # Masked self-attention
    attn_output, _ = self.self_attn(x, x, x, tgt_mask)
    x = self.norm1(x + self.dropout1(attn_output))
    
    # Cross-attention with encoder output
    cross_output, _ = self.cross_attn(x, encoder_output,encoder_output, src_mask)
    x = self.norm2(x + self.dropout2(cross_output))
    
    # Feed-forward
    ff_output = self.feed_forward(x)
    x = self.norm3(x + self.dropout3(ff_output))
    
    return x


In [28]:
#Step 7
class Transformer(nn.Module):
 """
 Complete Transformer encoder-decoder model.
 """
 def __init__(self, src_vocab_size, tgt_vocab_size, d_model=128,
    num_heads=8, num_layers=3, d_ff=512,
    max_seq_length=10, dropout=0.1):
    super(Transformer, self).__init__()

    self.d_model = d_model
    self.max_seq_length = max_seq_length

    # Embedding layers
    self.src_embedding = nn.Embedding(src_vocab_size, d_model,padding_idx=PAD_token)
    self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model,padding_idx=PAD_token)

    # Positional encoding
    self.positional_encoding = PositionalEncoding(d_model,max_seq_length, dropout)

    # Encoder and decoder stacks
    self.encoder_layers = nn.ModuleList([
    EncoderLayer(d_model, num_heads, d_ff, dropout)
    for _ in range(num_layers)])

    self.decoder_layers = nn.ModuleList([
    DecoderLayer(d_model, num_heads, d_ff, dropout)
    for _ in range(num_layers)])

    # Output projection
    self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    self.dropout = nn.Dropout(dropout)

 def generate_mask(self, src, tgt):
    """
    Generate padding mask for source and causal mask for target.
    Returns:
    src_mask: [batch_size, 1, src_len] - masks padding in source
    tgt_mask: [batch_size, tgt_len, tgt_len] - masks future
    positions + padding
    """
    # Source padding mask
    src_mask = (src != PAD_token).unsqueeze(1) # [batch, 1, src_len]

    # Target padding mask
    tgt_mask = (tgt != PAD_token).unsqueeze(1) # [batch, 1, tgt_len]

    # Causal mask (no peeking into future)
    seq_len = tgt.size(1)
    nopeak_mask = torch.tril(torch.ones(seq_len,seq_len)).bool().to(tgt.device)
    nopeak_mask = nopeak_mask.unsqueeze(0) # [1, tgt_len, tgt_len]

    # Combine: element-wise AND
    tgt_mask = tgt_mask.unsqueeze(1) & nopeak_mask # [batch, 1,tgt_len, tgt_len]

    return src_mask, tgt_mask
 
 def forward(self, src, tgt):
    """
    Args:
    src: [batch_size, src_seq_len] - source token indices
    tgt: [batch_size, tgt_seq_len] - target token indices

    Returns:
    output: [batch_size, tgt_seq_len, tgt_vocab_size] - logits
    """
    src_mask, tgt_mask = self.generate_mask(src, tgt)

    # Embed and add positional encoding
    src_embedded = self.dropout(self.positional_encoding(
    self.src_embedding(src) * math.sqrt(self.d_model)))

    tgt_embedded = self.dropout(self.positional_encoding(
    self.tgt_embedding(tgt) * math.sqrt(self.d_model)))

    # Pass through encoder layers
    enc_output = src_embedded
    for layer in self.encoder_layers:
        enc_output = layer(enc_output, src_mask)

    # Pass through decoder layers
    dec_output = tgt_embedded
    for layer in self.decoder_layers:
        dec_output = layer(dec_output, enc_output, src_mask, tgt_mask)

    # Final linear projection to vocabulary
        output = self.fc_out(dec_output)
        
        return output
    



# Model hyperparameters
d_model = 128
num_heads = 8
num_layers = 3
d_ff = 512
dropout = 0.1
transformer = Transformer(
 src_vocab_size=input_lang.n_words,
 tgt_vocab_size=output_lang.n_words,
 d_model=d_model,
 num_heads=num_heads,
 num_layers=num_layers,
 d_ff=d_ff,
 max_seq_length=MAX_LENGTH,
 dropout=dropout
).to(device)
print(f"Transformer parameters: {sum(p.numel() for p in
transformer.parameters()):,}")


Transformer parameters: 2,746,159


In [29]:
#Part 3
class LSTMEncoder(nn.Module):
 def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.1):
    super(LSTMEncoder, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers

    self.embedding = nn.Embedding(input_size, hidden_size,padding_idx=PAD_token)
    self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers,batch_first=True, dropout=dropout)
    self.dropout = nn.Dropout(dropout)

def forward(self, x):
 # x: [batch_size, seq_len]
 embedded = self.dropout(self.embedding(x))
 outputs, (hidden, cell) = self.lstm(embedded)
 # outputs: [batch_size, seq_len, hidden_size]
 # hidden: [num_layers, batch_size, hidden_size]
 return outputs, hidden, cell

class LSTMDecoder(nn.Module):
 def __init__(self, output_size, hidden_size, num_layers=2,dropout=0.1):
    super(LSTMDecoder, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.output_size = output_size

    self.embedding = nn.Embedding(output_size, hidden_size,padding_idx=PAD_token)
    self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers,batch_first=True, dropout=dropout)
    self.fc = nn.Linear(hidden_size, output_size)
    self.dropout = nn.Dropout(dropout)

 def forward(self, x, hidden, cell):
    # x: [batch_size, 1]
    embedded = self.dropout(self.embedding(x))
    output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
    prediction = self.fc(output.squeeze(1))
    return prediction, hidden, cell

class Seq2SeqLSTM(nn.Module):
 def __init__(self, src_vocab_size, tgt_vocab_size, hidden_size,num_layers=2, dropout=0.1):
    super(Seq2SeqLSTM, self).__init__()
    self.encoder = LSTMEncoder(src_vocab_size, hidden_size,num_layers, dropout)
    self.decoder = LSTMDecoder(tgt_vocab_size, hidden_size,num_layers, dropout)

 def forward(self, src, tgt, teacher_forcing_ratio=0.5):
    # src: [batch_size, src_len]
    # tgt: [batch_size, tgt_len]
    batch_size = src.size(0)
    tgt_len = tgt.size(1)
    tgt_vocab_size = self.decoder.output_size

    # Encoder
    encoder_outputs, hidden, cell = self.encoder(src)

    # Decoder - step by step
    outputs = torch.zeros(batch_size, tgt_len,tgt_vocab_size).to(src.device)

    # First input to decoder is SOS token
    decoder_input = tgt[:, 0].unsqueeze(1) # [batch, 1]

    for t in range(1, tgt_len):
       prediction, hidden, cell = self.decoder(decoder_input, hidden,cell)
       outputs[:, t, :] = prediction

       # Teacher forcing
       teacher_force = torch.rand(1).item() < teacher_forcing_ratio
       decoder_input = tgt[:, t].unsqueeze(1) if teacher_force else prediction.argmax(1).unsqueeze(1)

    return outputs

hidden_size = 128
num_layers = 2
lstm_model = Seq2SeqLSTM(
 src_vocab_size=input_lang.n_words,
 tgt_vocab_size=output_lang.n_words,
 hidden_size=hidden_size,
 num_layers=num_layers,
 dropout=0.1
).to(device)
print(f"LSTM parameters: {sum(p.numel() for p in
lstm_model.parameters()):,}")

LSTM parameters: 1,885,999


In [30]:
# Part 4 | Step 1

def train_epoch(model, dataloader, optimizer, criterion, model_type: str = 'transformer'):
    model.train()
    total_loss = 0

    for src, tgt in dataloader:
        optimizer.zero_grad()

        if model_type == 'transformer':
            # Transformer expects tgt input without last token
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            # Forward pass
            output = model(src, tgt_input)

            # Reshape for loss calculation
            output = output.reshape(-1, output.size(-1))
            tgt_output = tgt_output.reshape(-1)

        else:  # LSTM
            output = model(src, tgt)
            output = output[:, 1:, :].reshape(-1, output.size(-1))
            tgt_output = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt_output)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, criterion, model_type: str = 'transformer'):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in dataloader:
            if model_type == 'transformer':
                tgt_input = tgt[:, :-1]
                tgt_output = tgt[:, 1:]

                output = model(src, tgt_input)

                output = output.reshape(-1, output.size(-1))
                tgt_output = tgt_output.reshape(-1)

            else:
                output = model(src, tgt, teacher_forcing_ratio=0.0)

                output = output[:, 1:, :].reshape(-1, output.size(-1))
                tgt_output = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt_output)
            total_loss += loss.item()

    return total_loss / len(dataloader)


def train_model(
    model,
    train_loader,
    val_loader,
    num_epochs: int,
    learning_rate: float,
    model_type: str = 'transformer',
    model_name: str = 'Model'
):
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.98),
        eps=1e-9
    )

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_token)

    train_losses = []
    val_losses = []
    epoch_times = []

    print(f"\n{'=' * 60}")
    print(f"Training {model_name}")
    print(f"{'=' * 60}")

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            model_type
        )

        val_loss = evaluate(
            model,
            val_loader,
            criterion,
            model_type
        )

        epoch_time = time.time() - start_time
        epoch_times.append(epoch_time)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if epoch % 5 == 0:
            print(
                f"Epoch {epoch:2d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Time: {epoch_time:.2f}s"
            )

    avg_time = np.mean(epoch_times)

    print(f"\nAverage time per epoch: {avg_time:.2f}s")
    print(f"Final train loss: {train_losses[-1]:.4f}")
    print(f"Final val loss: {val_losses[-1]:.4f}")

    return train_losses, val_losses, epoch_times

In [31]:
#Step 2
# Create train/val split
from torch.utils.data import random_split

# Assume you have train_dataloader from Part 1
full_dataset = train_dataloader.dataset
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size,val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")



Train samples: 10300
Val samples: 1145


In [35]:
num_epochs = 30
learning_rate = 0.0001

# Train Transformer
transformer_losses, transformer_val_losses, transformer_times = train_model(
    transformer,
    train_loader,
    val_loader,
    num_epochs,
    learning_rate,
    model_type='transformer',
    model_name='Transformer'
)

# Train LSTM
lstm_losses, lstm_val_losses, lstm_times = train_model(
    lstm_model,
    train_loader,
    val_loader,
    num_epochs,
    learning_rate,
    model_type='lstm',
    model_name='LSTM Seq2Seq'
)


Training Transformer


ValueError: too many values to unpack (expected 4)